# Cue-balanced 8B development extraction

Run this notebook on the Google Colab website with an **A100 GPU**. It mounts Google Drive and extracts the unchanged fitting data, both already-spent control banks, and the development-only cue-context augmentation in one unquantized bfloat16 model session. Only mean layer 6 is stored. No probe is fitted, no confirmatory bank is consumed, and naturalistic transfer is never loaded.

Place the seven files listed in `CUE_BALANCED_DEVELOPMENT_HANDOFF.md` directly in `MyDrive/lang-ea/`. The model cells are independent. Completed, verified output is skipped after a disconnect.

In [ ]:
%pip install -q transformers accelerate sentencepiece

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, torch

BATCH_SIZE = 8
SKIP_COMPLETE = True
SWALLOW = 'tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3'
META = 'unsloth/Meta-Llama-3.1-8B-Instruct'
MODELS = [SWALLOW, META]
EXPECTED_EXTRACTOR = '520411bf9dead0e72ef6ccc1a7be99bb165dbb51db19911a0665830e1be5ec3f'
EXPECTED_ITEMS = '97796dcf3f54918cb7b55c009f6458832fcf75d880df50252ffdf8415c6bc84e'
EXPECTED_BANK1 = 'fbd320168c38e14321e50cab1872144a3d32e171d4a56afca63dcb6f782131b9'
EXPECTED_BANK2 = '0f93681d626c2e4d6d7c91acf9c8301f42f1203f45136d32441a2ef68de654fb'
EXPECTED_AUGMENTATION = 'ce7a41bb44b6ede12dee5455b0e8a586d06f15feb491ce4c91e16fdf74bb2b0c'
EXPECTED_TRIGGER = 'a7fd030bd0d3220d61d4c95a2e67cb91d2b6c033ff6c5a3e8c8dc5219c90781f'

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_DIR = Path('/content/drive/MyDrive/lang-ea')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTOR = DRIVE_DIR / 'extract_cue_balanced_development_8b.py'
HELPER = DRIVE_DIR / 'extract_confirmatory_8b.py'
ITEMS = DRIVE_DIR / 'probe_items.jsonl'
BANK1 = DRIVE_DIR / 'format_matched_controls.jsonl'
BANK2 = DRIVE_DIR / 'cue_invariant_confirmatory_controls.jsonl'
AUGMENTATION = DRIVE_DIR / 'cue_balanced_development_augmentation.jsonl'
TRIGGER = DRIVE_DIR / 'cue_balanced_development.json'
required = [EXTRACTOR, HELPER, ITEMS, BANK1, BANK2, AUGMENTATION, TRIGGER]
missing = [path.name for path in required if not path.is_file()]
if missing: raise FileNotFoundError(f'Missing from {DRIVE_DIR}: {missing}')
file_sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
assert file_sha256(EXTRACTOR) == EXPECTED_EXTRACTOR, 'Extractor upload has the wrong SHA-256.'
WORKDIR = EXTRACTOR.parent
OUTPUT = DRIVE_DIR / 'cue_balanced_development_8b'
assert torch.cuda.is_available(), 'Select the A100 GPU runtime.'
gpu = torch.cuda.get_device_properties(0)
assert 'A100' in gpu.name.upper(), f'A100 required; found {gpu.name}'
assert torch.cuda.is_bf16_supported(), 'This runtime does not report bfloat16 support.'
print(f'GPU: {gpu.name}; output: {OUTPUT}')

In [ ]:
# Hash and structure preflight; this does not load a model or activations.
subprocess.run([sys.executable, str(EXTRACTOR), '--items', str(ITEMS), '--bank1', str(BANK1), '--bank2', str(BANK2), '--augmentation', str(AUGMENTATION), '--trigger-result', str(TRIGGER), '--check-only'], cwd=WORKDIR, check=True)

In [ ]:
def model_dir(model): return OUTPUT / model.split('/')[-1]
def load_tensor(path):
    try: return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError: return torch.load(path, map_location='cpu')
def verify(model):
    out = model_dir(model)
    cfg = json.loads((out / 'config.json').read_text(encoding='utf-8'))
    assert cfg['analysis'] == 'cue_balanced_augmented_development_v1'
    assert cfg['development_only'] is True and cfg['model'] == model
    assert cfg['load_in_4bit'] is False and cfg['model_load_dtype'] == 'bfloat16'
    assert 'A100' in cfg['accelerator'].upper()
    assert cfg['items_sha256'] == EXPECTED_ITEMS
    assert cfg['bank1_sha256'] == EXPECTED_BANK1
    assert cfg['bank2_sha256'] == EXPECTED_BANK2
    assert cfg['augmentation_sha256'] == EXPECTED_AUGMENTATION
    assert cfg['trigger_result_sha256'] == EXPECTED_TRIGGER
    assert cfg['cell'] == {'position': 'mean', 'layer': 6}
    for directory, n in [(out, 1200), (out/'bank1', 256), (out/'bank2', 256), (out/'augmentation', 1536)]:
        assert len((directory/'meta.jsonl').read_text(encoding='utf-8').splitlines()) == n
        tensor = load_tensor(directory/'acts_layer6_mean.pt')
        assert tuple(tensor.shape) == (n, 4096), tensor.shape
        del tensor
    print(f'VERIFIED: {model} -> {out}')
def run_model(model, force=False):
    if SKIP_COMPLETE and not force:
        try:
            verify(model); print('Complete output exists; skipping.'); return
        except (AssertionError, FileNotFoundError, KeyError, json.JSONDecodeError):
            pass
    command = [sys.executable, str(EXTRACTOR), '--items', str(ITEMS), '--bank1', str(BANK1), '--bank2', str(BANK2), '--augmentation', str(AUGMENTATION), '--trigger-result', str(TRIGGER), '--out', str(OUTPUT), '--models', model, '--batch-size', str(BATCH_SIZE)]
    OUTPUT.mkdir(parents=True, exist_ok=True)
    log_path = OUTPUT / f"{model.split('/')[-1]}.log"
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=WORKDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Extraction failed with exit code {returncode}. Full log: {log_path}')
    verify(model)

## 1. Swallow 8B

In [ ]:
run_model(SWALLOW)

## 2. Meta-Llama 8B

In [ ]:
run_model(META)

In [ ]:
for model in MODELS: verify(model)
print('BOTH DEVELOPMENT EXTRACTIONS COMPLETE. Copy both model directories back into probe_check/out/cue_balanced_development_8b/.')